# 02. Cleaning & Preprocessing

Notebook này thực hiện quá trình làm sạch và chuẩn hóa dữ liệu trước khi tiến hành phân tích khám phá dữ liệu và xây dựng mô hình học máy.

Các bước xử lý bao gồm:

- Tạo bản sao dữ liệu làm việc
- Ghi nhận các quyết định làm sạch
- Xử lý dữ liệu trùng lặp
- Xử lý giá trị khuyết
- Chuẩn hóa kiểu dữ liệu
- Chuẩn hóa biến nghiên cứu
- Xây dựng biến mục tiêu phục vụ Machine Learning
- Kiểm tra ngoại lệ
- Đánh giá chất lượng dữ liệu sau làm sạch
- Lưu bộ dữ liệu đã xử lý

Output của notebook là bộ dữ liệu đã làm sạch và sẵn sàng cho các bước nghiên cứu tiếp theo.

## 0. Set Up

Phần này khởi tạo môi trường làm việc và import các thư viện cần thiết cho quá trình làm sạch dữ liệu.

In [47]:
# Import các thư viện cần thiết

import warnings

import numpy as np
import pandas as pd

from scipy import stats

warnings.filterwarnings("ignore")

In [48]:
# Thiết lập các tùy chọn hiển thị

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.3f}".format)

RANDOM_STATE = 42

print("Thiết lập môi trường hoàn tất.")

Thiết lập môi trường hoàn tất.


# 1. Data Preparation

Phần này chuẩn bị bộ dữ liệu cho toàn bộ quá trình làm sạch.

Đầu tiên, bộ dữ liệu gốc được tải vào môi trường làm việc. Sau đó, một bản sao của dữ liệu được tạo ra nhằm đảm bảo dữ liệu ban đầu luôn được giữ nguyên trong suốt quá trình nghiên cứu.

Tất cả các thao tác làm sạch và tiền xử lý trong notebook sẽ được thực hiện trên bản sao này.

In [49]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]

input_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "international_dataset"
    / "digital_burnout_productivity_dataset_5M.csv"
)

In [50]:
# Đọc bộ dữ liệu

raw_df = pd.read_csv(
    input_path
)

print("Đã tải bộ dữ liệu thành công.")

Đã tải bộ dữ liệu thành công.


In [51]:
# Hiển thị kích thước bộ dữ liệu

print("Thông tin bộ dữ liệu")
print()

print(f"Số dòng: {raw_df.shape[0]:,}")
print(f"Số cột: {raw_df.shape[1]}")

Thông tin bộ dữ liệu

Số dòng: 5,000,000
Số cột: 34


In [52]:
# Hiển thị một số bản ghi đầu tiên

raw_df.head()

,user_id,age,occupation,work_mode,device_usage_type,daily_screen_time,social_media_hours,doomscrolling_duration,app_switch_frequency,notification_count,smartphone_unlocks,late_night_device_usage,focus_sessions,deep_work_hours,distraction_frequency,task_completion_rate,concentration_score,sleep_hours,sleep_quality,caffeine_intake,physical_activity,stress_level,workspace_quality,meeting_hours,internet_stability,remote_work_days,motivation_level,mental_fatigue,emotional_exhaustion,work_satisfaction,mental_state,burnout_risk,productivity_score,productivity_category
0,1,56,Content Creator,Office,Entertainment-Centric,8.800,5.000,1.200,41,112,49,1,3,6.000,67,92,2,5.900,10,6,1.600,10,5,2.800,3,4,8.000,10,4,8,Balanced,46,100,High
1,2,46,Student,Hybrid,Work-Centric,10.300,2.200,2.400,119,168,153,1,9,4.600,75,74,3,5.600,7,5,1.100,5,10,3.200,7,6,5.000,7,9,7,Balanced,57,96,High
2,3,32,Software Engineer,Remote,Balanced,6.500,4.600,1.000,121,199,234,1,5,NaN,70,96,7,5.500,8,6,1.100,4,1,2.300,9,6,8.000,5,2,6,Balanced,29,79,High
3,4,25,Designer,Office,Balanced,9.600,1.200,0.100,85,122,177,1,9,2.900,107,60,3,6.100,5,1,1.700,1,4,3.800,10,5,6.000,4,5,3,Burnout,57,63,Medium
4,5,38,Analyst,Hybrid,Work-Centric,13.300,1.600,1.900,221,73,91,1,9,2.800,72,73,9,7.900,1,3,1.800,4,8,3.000,1,1,4.000,7,9,7,Focused,64,89,High


In [53]:
# Hiển thị thông tin tổng quát

raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 34 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   user_id                  int64  
 1   age                      int64  
 2   occupation               object 
 3   work_mode                object 
 4   device_usage_type        object 
 5   daily_screen_time        float64
 6   social_media_hours       float64
 7   doomscrolling_duration   float64
 8   app_switch_frequency     int64  
 9   notification_count       int64  
 10  smartphone_unlocks       int64  
 11  late_night_device_usage  int64  
 12  focus_sessions           int64  
 13  deep_work_hours          float64
 14  distraction_frequency    int64  
 15  task_completion_rate     int64  
 16  concentration_score      int64  
 17  sleep_hours              float64
 18  sleep_quality            int64  
 19  caffeine_intake          int64  
 20  physical_activity        float64
 21  stress_l

In [54]:
# Tạo bản sao của bộ dữ liệu

df = raw_df.copy()

print("Đã tạo bản sao dữ liệu.")

Đã tạo bản sao dữ liệu.


In [55]:
# Kiểm tra kích thước dữ liệu

print("Kích thước dữ liệu làm việc")
print()

print(f"Số dòng: {df.shape[0]:,}")
print(f"Số cột: {df.shape[1]}")

Kích thước dữ liệu làm việc

Số dòng: 5,000,000
Số cột: 34


In [56]:
# Kiểm tra việc sao chép dữ liệu

print("Kiểm tra dữ liệu")

print()

print(
    "Bản sao được tạo thành công."
    if id(raw_df) != id(df)
    else
    "Không thể tạo bản sao dữ liệu."
)

Kiểm tra dữ liệu

Bản sao được tạo thành công.


In [57]:
# Hiển thị danh sách các biến

print("Danh sách biến trong bộ dữ liệu")
print()

for column in df.columns:
    print(column)

Danh sách biến trong bộ dữ liệu

user_id
age
occupation
work_mode
device_usage_type
daily_screen_time
social_media_hours
doomscrolling_duration
app_switch_frequency
notification_count
smartphone_unlocks
late_night_device_usage
focus_sessions
deep_work_hours
distraction_frequency
task_completion_rate
concentration_score
sleep_hours
sleep_quality
caffeine_intake
physical_activity
stress_level
workspace_quality
meeting_hours
internet_stability
remote_work_days
motivation_level
mental_fatigue
emotional_exhaustion
work_satisfaction
mental_state
burnout_risk
productivity_score
productivity_category


# 2. Data Cleaning

Phần này thực hiện các bước làm sạch dữ liệu nhằm nâng cao chất lượng bộ dữ liệu trước khi tiến hành phân tích và xây dựng mô hình.

Quá trình làm sạch bao gồm:

- Ghi nhận các quyết định làm sạch
- Loại bỏ bản ghi trùng lặp
- Xử lý giá trị khuyết
- Chuẩn hóa kiểu dữ liệu

Mỗi quyết định đều được ghi nhận nhằm đảm bảo tính minh bạch và khả năng tái lập của nghiên cứu.

## 2.1 Cleaning decision log

Phần này ghi nhận toàn bộ các quyết định làm sạch dữ liệu sẽ được áp dụng trong nghiên cứu.

Cleaning Decision Log đóng vai trò như nhật ký xử lý dữ liệu, giúp theo dõi các thao tác tiền xử lý và tăng tính minh bạch của quy trình nghiên cứu.

In [58]:
# Khởi tạo nhật ký làm sạch dữ liệu

cleaning_log = []

In [59]:
# Xây dựng hàm ghi nhận quyết định làm sạch

def log_cleaning_step(step, action, reason):

    cleaning_log.append(
        {
            "Step": step,
            "Action": action,
            "Reason": reason
        }
    )

In [60]:
# Ghi nhận các bước làm sạch dự kiến

log_cleaning_step(
    step="Duplicate Records",
    action="Remove duplicated records",
    reason="Loại bỏ các bản ghi trùng lặp nhằm đảm bảo mỗi quan sát chỉ xuất hiện một lần."
)

log_cleaning_step(
    step="Missing Values",
    action="Handle missing values",
    reason="Xử lý các giá trị khuyết để nâng cao chất lượng dữ liệu."
)

log_cleaning_step(
    step="Data Types",
    action="Validate data types",
    reason="Chuẩn hóa kiểu dữ liệu phù hợp với từng biến nghiên cứu."
)

log_cleaning_step(
    step="Target Variable",
    action="Rename target variable",
    reason="Đổi tên biến burnout_risk thành burnout_score để thống nhất với ý nghĩa của biến nghiên cứu."
)

log_cleaning_step(
    step="Outlier Detection",
    action="Flag potential outliers",
    reason="Đánh dấu các giá trị ngoại lệ để phục vụ quá trình đánh giá chất lượng dữ liệu."
)

In [61]:
# Chuyển nhật ký làm sạch thành DataFrame

cleaning_log_df = pd.DataFrame(
    cleaning_log
)

# Hiển thị nhật ký làm sạch

print("Nhật ký các quyết định làm sạch dữ liệu")

cleaning_log_df

Nhật ký các quyết định làm sạch dữ liệu


,Step,Action,Reason
0,Duplicate Records,Remove duplicated records,Loại bỏ các bản ghi trùng lặp nhằm đảm bảo mỗi...
1,Missing Values,Handle missing values,Xử lý các giá trị khuyết để nâng cao chất lượn...
2,Data Types,Validate data types,Chuẩn hóa kiểu dữ liệu phù hợp với từng biến n...
3,Target Variable,Rename target variable,Đổi tên biến burnout_risk thành burnout_score ...
4,Outlier Detection,Flag potential outliers,Đánh dấu các giá trị ngoại lệ để phục vụ quá t...


## 2.2 Remove duplicate records

Phần này kiểm tra và loại bỏ các bản ghi trùng lặp trong bộ dữ liệu.

Việc loại bỏ dữ liệu trùng lặp giúp đảm bảo mỗi quan sát chỉ xuất hiện một lần, tránh ảnh hưởng đến quá trình phân tích thống kê và xây dựng mô hình học máy.

In [62]:
# Kiểm tra số lượng bản ghi trước khi loại bỏ dữ liệu trùng lặp

rows_before = len(df)

print("Thông tin trước khi xử lý dữ liệu trùng lặp")
print()

print(f"Số bản ghi: {rows_before:,}")

Thông tin trước khi xử lý dữ liệu trùng lặp

Số bản ghi: 5,000,000


In [63]:
# Kiểm tra số lượng bản ghi trùng lặp

duplicate_count = df.duplicated().sum()

print("Kết quả kiểm tra dữ liệu trùng lặp")
print()

print(f"Số bản ghi trùng lặp: {duplicate_count:,}")

Kết quả kiểm tra dữ liệu trùng lặp

Số bản ghi trùng lặp: 0


In [64]:
# Loại bỏ các bản ghi trùng lặp

if duplicate_count > 0:

    df = df.drop_duplicates().reset_index(drop=True)

rows_after = len(df)

removed_duplicates = rows_before - rows_after

print("Hoàn tất xử lý dữ liệu trùng lặp")
print()

print(f"Số bản ghi đã loại bỏ: {removed_duplicates:,}")
print(f"Số bản ghi còn lại: {rows_after:,}")

Hoàn tất xử lý dữ liệu trùng lặp

Số bản ghi đã loại bỏ: 0
Số bản ghi còn lại: 5,000,000


In [65]:
# Cập nhật trạng thái nhật ký làm sạch

cleaning_log_df.loc[
    cleaning_log_df["Step"] == "Duplicate Records",
    "Status"
] = "Completed"

print("Đã cập nhật nhật ký làm sạch.")

Đã cập nhật nhật ký làm sạch.


In [66]:
# Hiển thị nhật ký làm sạch sau khi cập nhật

cleaning_log_df

,Step,Action,Reason,Status
0,Duplicate Records,Remove duplicated records,Loại bỏ các bản ghi trùng lặp nhằm đảm bảo mỗi...,Completed
1,Missing Values,Handle missing values,Xử lý các giá trị khuyết để nâng cao chất lượn...,NaN
2,Data Types,Validate data types,Chuẩn hóa kiểu dữ liệu phù hợp với từng biến n...,NaN
3,Target Variable,Rename target variable,Đổi tên biến burnout_risk thành burnout_score ...,NaN
4,Outlier Detection,Flag potential outliers,Đánh dấu các giá trị ngoại lệ để phục vụ quá t...,NaN


## 2.3 Handle missing values

Phần này kiểm tra và xử lý các giá trị khuyết trong bộ dữ liệu nhằm đảm bảo dữ liệu đầy đủ trước khi phân tích và xây dựng mô hình.

Đối với các biến số, giá trị khuyết sẽ được thay thế bằng trung vị (Median). Đối với các biến phân loại, giá trị khuyết sẽ được thay thế bằng giá trị xuất hiện nhiều nhất (Mode). Phương pháp này giúp giảm ảnh hưởng của ngoại lệ và duy trì phân bố dữ liệu.

In [67]:
# Thống kê số lượng và tỷ lệ giá trị khuyết

missing_count = df.isna().sum()

missing_percentage = (
    missing_count / len(df) * 100
).round(2)

missing_summary = pd.DataFrame({

    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage

})

missing_summary = (
    missing_summary[
        missing_summary["Missing Count"] > 0
    ]
    .sort_values(
        by="Missing Count",
        ascending=False
    )
)

print("Thống kê giá trị khuyết")

display(missing_summary)

Thống kê giá trị khuyết


,Missing Count,Missing Percentage
social_media_hours,100000,2.000
deep_work_hours,100000,2.000
sleep_hours,100000,2.000
motivation_level,100000,2.000


In [68]:
# Xác định các biến số và biến phân loại

numerical_columns = df.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = df.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Thông tin các nhóm biến")
print()

print(f"Số biến số: {len(numerical_columns)}")
print(f"Số biến phân loại: {len(categorical_columns)}")

Thông tin các nhóm biến

Số biến số: 29
Số biến phân loại: 5


In [69]:
# Xử lý giá trị khuyết của các biến số

for column in numerical_columns:

    if missing_count[column] > 0:

        df[column] = df[column].fillna(
            df[column].median()
        )

print("Đã xử lý giá trị khuyết của các biến số.")

Đã xử lý giá trị khuyết của các biến số.


In [70]:
# Xử lý giá trị khuyết của các biến phân loại

for column in categorical_columns:

    if missing_count[column] > 0:

        mode_value = df[column].mode()

        if not mode_value.empty:

            df[column] = df[column].fillna(
                mode_value.iloc[0]
            )

print("Đã xử lý giá trị khuyết của các biến phân loại.")

Đã xử lý giá trị khuyết của các biến phân loại.


In [71]:
# Kiểm tra lại sau khi xử lý

remaining_missing = df.isna().sum().sum()

print("Kết quả sau khi xử lý giá trị khuyết")
print()

print(f"Tổng số giá trị khuyết còn lại: {remaining_missing:,}")

Kết quả sau khi xử lý giá trị khuyết

Tổng số giá trị khuyết còn lại: 0


In [72]:
# Cập nhật nhật ký làm sạch

cleaning_log_df.loc[
    cleaning_log_df["Step"] == "Missing Values",
    "Status"
] = "Completed"

print("Đã cập nhật nhật ký làm sạch.")

Đã cập nhật nhật ký làm sạch.


## 2.4 Validate data types

Phần này kiểm tra và chuẩn hóa kiểu dữ liệu của các biến nghiên cứu nhằm đảm bảo dữ liệu phù hợp cho quá trình phân tích và xây dựng mô hình.

Biến mục tiêu được đổi tên từ burnout_risk thành burnout_score nhằm phản ánh đúng bản chất là một thang điểm liên tục từ 0 đến 100.

Sau bước này, bộ dữ liệu đã được chuẩn hóa về kiểu dữ liệu và sẵn sàng cho quá trình kiểm tra chất lượng dữ liệu ở phần tiếp theo.

In [73]:
# Hiển thị kiểu dữ liệu của các biến

dtype_summary = pd.DataFrame({

    "Data Type": df.dtypes.astype(str)

})

print("Kiểu dữ liệu hiện tại")

display(dtype_summary)

Kiểu dữ liệu hiện tại


,Data Type
user_id,int64
age,int64
occupation,object
work_mode,object
device_usage_type,object
daily_screen_time,float64
social_media_hours,float64
doomscrolling_duration,float64
app_switch_frequency,int64
notification_count,int64


In [74]:
# Chuyển các biến dạng object sang category

object_columns = df.select_dtypes(
    include="object"
).columns.tolist()

for column in object_columns:

    df[column] = df[column].astype("category")

print("Đã chuẩn hóa các biến phân loại.")

Đã chuẩn hóa các biến phân loại.


In [75]:
# Đổi tên biến mục tiêu

if "burnout_risk" in df.columns:

    df.rename(
        columns={
            "burnout_risk": "burnout_score"
        },
        inplace=True
    )

print("Đã chuẩn hóa tên biến mục tiêu.")

Đã chuẩn hóa tên biến mục tiêu.


In [76]:
# Kiểm tra kiểu dữ liệu sau khi chuẩn hóa

dtype_summary = pd.DataFrame({

    "Data Type": df.dtypes.astype(str)

})

print("Kiểu dữ liệu sau khi chuẩn hóa")

display(dtype_summary)

Kiểu dữ liệu sau khi chuẩn hóa


,Data Type
user_id,int64
age,int64
occupation,category
work_mode,category
device_usage_type,category
daily_screen_time,float64
social_media_hours,float64
doomscrolling_duration,float64
app_switch_frequency,int64
notification_count,int64


In [77]:
cleaning_log_df.loc[
    cleaning_log_df["Step"] == "Data Types",
    "Status"
] = "Completed"

print("Đã cập nhật nhật ký làm sạch.")

Đã cập nhật nhật ký làm sạch.


# 3. Research Data Validation

Sau khi hoàn thành quá trình làm sạch dữ liệu, phần này kiểm tra lại chất lượng của bộ dữ liệu nhằm đảm bảo dữ liệu đáp ứng yêu cầu của nghiên cứu.

Quá trình kiểm tra bao gồm:

- Kiểm kê các biến nghiên cứu
- Đánh dấu các giá trị ngoại lệ
- Đánh giá chất lượng dữ liệu cuối cùng trước khi lưu bộ dữ liệu đã làm sạch

### 3.1 Research variable inventory

Phần này kiểm kê toàn bộ các biến nghiên cứu sau quá trình làm sạch dữ liệu nhằm xác nhận bộ dữ liệu vẫn giữ đầy đủ các biến cần thiết cho các bước phân tích tiếp theo.

In [78]:
# Tổng hợp thông tin các biến nghiên cứu

variable_inventory = pd.DataFrame({

    "Variable": df.columns,
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Unique Values": df.nunique()

})

print("Danh sách các biến nghiên cứu")

display(variable_inventory)

Danh sách các biến nghiên cứu


,Variable,Data Type,Missing Values,Unique Values
user_id,user_id,int64,0,5000000
age,age,int64,0,42
occupation,occupation,category,0,7
work_mode,work_mode,category,0,3
device_usage_type,device_usage_type,category,0,3
daily_screen_time,daily_screen_time,float64,0,171
social_media_hours,social_media_hours,float64,0,121
doomscrolling_duration,doomscrolling_duration,float64,0,80
app_switch_frequency,app_switch_frequency,int64,0,240
notification_count,notification_count,int64,0,380


In [79]:
# Thống kê số lượng biến theo kiểu dữ liệu

dtype_summary = (

    variable_inventory["Data Type"]
    .value_counts()
    .rename_axis("Data Type")
    .reset_index(name="Count")

)

print("Thống kê kiểu dữ liệu")

display(dtype_summary)

Thống kê kiểu dữ liệu


,Data Type,Count
0,int64,21
1,float64,8
2,category,5


In [80]:
# Hiển thị thông tin tổng quan

print("Tổng quan bộ biến nghiên cứu")
print()

print(f"Tổng số biến: {df.shape[1]}")
print(f"Tổng số biến số: {len(df.select_dtypes(include='number').columns)}")
print(f"Tổng số biến phân loại: {len(df.select_dtypes(exclude='number').columns)}")

Tổng quan bộ biến nghiên cứu

Tổng số biến: 34
Tổng số biến số: 29
Tổng số biến phân loại: 5


### 3.2 Outlier flagging

Phần này phát hiện và thống kê các giá trị ngoại lệ trong các biến số bằng phương pháp Interquartile Range (IQR).

Các giá trị ngoại lệ chỉ được đánh dấu để phục vụ phân tích ở các notebook tiếp theo và không bị loại bỏ khỏi bộ dữ liệu.

In [81]:
# Lấy danh sách các biến số

numerical_columns = [

    column

    for column in df.select_dtypes(include=["number"]).columns

    if column not in [
        "burnout_score",
        "productivity_score"
    ]

]

In [82]:
# Khởi tạo danh sách lưu kết quả phát hiện ngoại lệ

outlier_summary = []

In [83]:
# Kiểm tra ngoại lệ bằng phương pháp IQR

for column in numerical_columns:

    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = (
        (
            (df[column] < lower_bound)
            |
            (df[column] > upper_bound)
        )
    ).sum()

    outlier_summary.append({

        "Variable": column,
        "Outlier Count": outlier_count,
        "Outlier Percentage": round(
            outlier_count / len(df) * 100,
            2
        )

    })

In [84]:
# Chuyển kết quả thành DataFrame

outlier_summary = pd.DataFrame(
    outlier_summary
)

outlier_summary = outlier_summary.sort_values(

    by="Outlier Count",
    ascending=False

)

print("Thống kê giá trị ngoại lệ")

display(outlier_summary)

Thống kê giá trị ngoại lệ


,Variable,Outlier Count,Outlier Percentage
2,daily_screen_time,34754,0.700
10,deep_work_hours,17362,0.350
3,social_media_hours,17325,0.350
20,meeting_hours,17087,0.340
4,doomscrolling_duration,16924,0.340
17,physical_activity,15999,0.320
0,user_id,0,0.000
16,caffeine_intake,0,0.000
25,emotional_exhaustion,0,0.000
24,mental_fatigue,0,0.000


In [85]:
# Cập nhật nhật ký làm sạch

cleaning_log_df.loc[
    cleaning_log_df["Step"] == "Outlier Detection",
    "Status"
] = "Completed"

print("Đã cập nhật nhật ký làm sạch.")

Đã cập nhật nhật ký làm sạch.


### 3.3 Final quality check

Phần này đánh giá chất lượng cuối cùng của bộ dữ liệu sau khi hoàn thành toàn bộ quá trình làm sạch và chuẩn hóa nhằm đảm bảo dữ liệu sẵn sàng cho các bước phân tích tiếp theo.

In [86]:
# Tổng hợp các chỉ số chất lượng dữ liệu

quality_summary = pd.DataFrame({

    "Metric": [

        "Total Samples",
        "Total Variables",
        "Duplicate Records",
        "Missing Values"

    ],

    "Value": [

        len(df),
        df.shape[1],
        df.duplicated().sum(),
        df.isna().sum().sum()

    ]

})

print("Kết quả kiểm tra chất lượng dữ liệu")

display(quality_summary)

Kết quả kiểm tra chất lượng dữ liệu


,Metric,Value
0,Total Samples,5000000
1,Total Variables,34
2,Duplicate Records,0
3,Missing Values,0


In [87]:
# Hiển thị thông tin bộ dữ liệu cuối cùng

print("Tổng kết bộ dữ liệu sau khi làm sạch")
print()

print(f"Số bản ghi: {len(df):,}")
print(f"Số biến: {df.shape[1]}")
print(f"Tổng giá trị khuyết: {df.isna().sum().sum():,}")
print(f"Tổng dữ liệu trùng lặp: {df.duplicated().sum():,}")

Tổng kết bộ dữ liệu sau khi làm sạch

Số bản ghi: 5,000,000
Số biến: 34
Tổng giá trị khuyết: 0
Tổng dữ liệu trùng lặp: 0


# 4. Target Engineering

Section này xây dựng biến mục tiêu phục vụ bài toán phân loại Digital Burnout. Dựa trên thang điểm Burnout Score (0–100), nghiên cứu chuyển đổi điểm số liên tục thành ba mức độ Burnout gồm Low, Moderate và High nhằm phù hợp với các mô hình phân loại được sử dụng ở các bước tiếp theo.

In [88]:
# Xây dựng mức độ Digital Burnout

def classify_burnout_level(score):

    if score <= 33:
        return "Low"
    elif score <= 66:
        return "Moderate"
    else:
        return "High"


# Tạo biến mục tiêu

df["burnout_level"] = (

    df["burnout_score"]
    .apply(classify_burnout_level)

)

print(
    "Đã xây dựng biến burnout_level."
)

display(

    df["burnout_level"]
    
    .value_counts()
    
    .rename_axis(
        "Mức độ Digital Burnout"
    )

    .reset_index(
        name="Số lượng"
    )

)

Đã xây dựng biến burnout_level.


,Mức độ Digital Burnout,Số lượng
0,Moderate,2643212
1,Low,1279885
2,High,1076903


# 5. Save Cleaned Dataset

Section này lưu bộ dữ liệu đã hoàn thành quá trình làm sạch, kiểm tra chất lượng và xây dựng biến mục tiêu. Bộ dữ liệu sẽ được sử dụng trực tiếp cho các Notebook tiếp theo trong quy trình nghiên cứu.

In [89]:
# Khai báo đường dẫn lưu bộ dữ liệu

output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "international_dataset"
    / "digital_burnout_cleaned.csv"
)

In [90]:
# Lưu bộ dữ liệu đã làm sạch

df.to_csv(

    output_path,
    index=False

)

print("Đã lưu bộ dữ liệu thành công.")

Đã lưu bộ dữ liệu thành công.


In [91]:
# Kiểm tra lại tệp sau khi lưu

saved_df = pd.read_csv(
    output_path,
    nrows=5
)

print("Kiểm tra tệp đã lưu")
print()

display(saved_df)

Kiểm tra tệp đã lưu



,user_id,age,occupation,work_mode,device_usage_type,daily_screen_time,social_media_hours,doomscrolling_duration,app_switch_frequency,notification_count,smartphone_unlocks,late_night_device_usage,focus_sessions,deep_work_hours,distraction_frequency,task_completion_rate,concentration_score,sleep_hours,sleep_quality,caffeine_intake,physical_activity,stress_level,workspace_quality,meeting_hours,internet_stability,remote_work_days,motivation_level,mental_fatigue,emotional_exhaustion,work_satisfaction,mental_state,burnout_score,productivity_score,productivity_category,burnout_level
0,1,56,Content Creator,Office,Entertainment-Centric,8.800,5.000,1.200,41,112,49,1,3,6.000,67,92,2,5.900,10,6,1.600,10,5,2.800,3,4,8.000,10,4,8,Balanced,46,100,High,Moderate
1,2,46,Student,Hybrid,Work-Centric,10.300,2.200,2.400,119,168,153,1,9,4.600,75,74,3,5.600,7,5,1.100,5,10,3.200,7,6,5.000,7,9,7,Balanced,57,96,High,Moderate
2,3,32,Software Engineer,Remote,Balanced,6.500,4.600,1.000,121,199,234,1,5,3.000,70,96,7,5.500,8,6,1.100,4,1,2.300,9,6,8.000,5,2,6,Balanced,29,79,High,Low
3,4,25,Designer,Office,Balanced,9.600,1.200,0.100,85,122,177,1,9,2.900,107,60,3,6.100,5,1,1.700,1,4,3.800,10,5,6.000,4,5,3,Burnout,57,63,Medium,Moderate
4,5,38,Analyst,Hybrid,Work-Centric,13.300,1.600,1.900,221,73,91,1,9,2.800,72,73,9,7.900,1,3,1.800,4,8,3.000,1,1,4.000,7,9,7,Focused,64,89,High,Moderate
